In [4]:
import os
import numpy as np, h5py
from glob import glob
from astropy.table import Table, vstack
from astropy.coordinates import SkyCoord
import astropy.units as u

# ---- config ----
RA0, DEC0 = 266.405, -28.936    # Galactic Center in ICRS (l=0, b=0)
RADIUS_DEG = 1.0                # cone radius
OUT_FITS   = "gc_sightline.fits"
# NOTE: no parallax / S/N cut is applied anymore. plx_over_err is still
# recorded as a column so you can filter on it later if you want.
# stellar_params_* column order: 0=Teff(1e3 K), 1=[Fe/H], 2=logg, 3=ext, 4=parallax(mas)

gc = SkyCoord(RA0*u.deg, DEC0*u.deg)
DATA_DIR = "Gaia XP"   # folder holding the .h5 files
files = sorted(glob(os.path.join(DATA_DIR, "stellar_params_catalog_*.h5")))
print(f"Found {len(files)} file(s): {files}")

pieces = []
for loc in files:
    with h5py.File(loc, "r") as f:
        ra  = f["ra"][:]
        dec = f["dec"][:]
        qf  = f["quality_flags"][:]

        # cheap box pre-cut (+ quality) so we don't run separation on 22M points
        dra = RADIUS_DEG / np.cos(np.radians(DEC0)) + 0.05
        cand = (np.abs(dec - DEC0) < RADIUS_DEG + 0.05) & \
               (np.abs(ra  - RA0) < dra) & (qf < 8)
        if not cand.any():
            print(f"{loc}: 0 candidates, skipping")
            continue

        # precise angular cut on the survivors only
        idx = np.nonzero(cand)[0]
        sep = SkyCoord(ra[idx]*u.deg, dec[idx]*u.deg).separation(gc).deg
        idx = idx[sep < RADIUS_DEG]
        if idx.size == 0:
            print(f"{loc}: 0 within {RADIUS_DEG} deg, skipping")
            continue

        # read the big arrays only for these rows (idx is sorted -> h5py OK)
        est = f["stellar_params_est"][idx, :]
        err = f["stellar_params_err"][idx, :]
        sid = f["gdr3_source_id"][idx]

    plx, eplx = est[:, 4], err[:, 4]
    # recorded for reference only -- NOT used to cut. Suppress divide/NaN warnings.
    with np.errstate(divide="ignore", invalid="ignore"):
        sn = np.abs(plx / eplx)

    # No parallax cut: every star in the cone that passed the box + quality
    # + separation cuts is kept, including negative, zero-error, and NaN parallax.
    t = Table()
    t["gdr3_source_id"] = sid
    t["ra"]  = ra[idx]
    t["dec"] = dec[idx]
    t["parallax"]      = plx                  # mas
    t["parallax_err"]  = eplx                 # mas
    t["ext"]           = est[:, 3]            # extinction
    t["ext_err"]       = err[:, 3]
    t["teff_K"]        = est[:, 0] * 1000     # native units are 1e3 K
    t["feh"]           = est[:, 1]
    t["logg"]          = est[:, 2]
    t["plx_over_err"]  = sn                   # for optional later filtering
    pieces.append(t)
    print(f"{loc}: kept {len(t)} GC stars (no parallax cut)")

if pieces:
    gc_stars = vstack(pieces)
    gc_stars.write(OUT_FITS, overwrite=True)
    print(f"\nWrote {len(gc_stars)} stars to {OUT_FITS}")
    print(gc_stars["parallax", "parallax_err", "ext", "ext_err"][:5])
else:
    print("\nNo GC stars found. None of the local files cover the Galactic "
          "Center — download the chunk the file-finder flagged and rerun.")

Found 5 file(s): ['Gaia XP/stellar_params_catalog_00.h5', 'Gaia XP/stellar_params_catalog_01.h5', 'Gaia XP/stellar_params_catalog_02.h5', 'Gaia XP/stellar_params_catalog_03.h5', 'Gaia XP/stellar_params_catalog_04.h5']
Gaia XP/stellar_params_catalog_00.h5: 0 candidates, skipping
Gaia XP/stellar_params_catalog_01.h5: 0 candidates, skipping
Gaia XP/stellar_params_catalog_02.h5: 0 candidates, skipping
Gaia XP/stellar_params_catalog_03.h5: kept 54602 GC stars (no parallax cut)
Gaia XP/stellar_params_catalog_04.h5: 0 candidates, skipping

Wrote 54602 stars to gc_sightline.fits
 parallax  parallax_err    ext       ext_err   
---------- ------------ ---------- ------------
 0.7040203  0.022521514 0.92221534  0.008815412
 3.4449449  0.017066715 0.19530182 0.0058992337
 0.3590552   0.10226834   0.793274   0.08245344
 1.1571022  0.039130934 0.22961904  0.018839965
0.45584366  0.024301944 0.79604214  0.012416963


In [2]:
import os
print("cwd:", os.getcwd())
print(glob("**/stellar_params_catalog_*.h5", recursive=True))

cwd: /home/dtfrake/hmc-dust
['Gaia XP/stellar_params_catalog_00.h5', 'Gaia XP/stellar_params_catalog_03.h5', 'Gaia XP/stellar_params_catalog_01.h5', 'Gaia XP/stellar_params_catalog_04.h5', 'Gaia XP/stellar_params_catalog_02.h5']
